# Первая задача: метод установления и Монте-Карло

Запускайте ячейки сверху вниз. Вычисления используют стандартные модули Python и обычные циклы. Для графиков нужны Matplotlib и NumPy: при необходимости выполните в отдельной ячейке `%pip install matplotlib numpy`.

Общие функции сетки, копирования, прогонки, ошибок, невязки и построения поверхности находятся в `numerical_common.py`. Их можно импортировать в следующих заданиях. Метод установления находится в `settling.py`, Монте-Карло — в `monte_carlo.py`. В ноутбуке оставлены постановка, формулы и вызовы алгоритмов. Все три модуля используют обычные циклы. Источник и граничные значения передаются как параметры; текущие алгоритмы предназначены для уравнения Пуассона на единичном квадрате. Для нелинейных и трёхмерных задач сами методы потребуется расширить.

Текстовый вывод вычислительных ячеек помещён в HTML-блоки: нажмите на строку с треугольником, чтобы раскрыть или свернуть результат. Это не зависит от поддержки сворачивания заголовков. Заключение также оформлено отдельным сворачиваемым блоком.

Каждый заголовок вынесен в отдельную Markdown-ячейку. В JupyterLab и Notebook 7 раздел сворачивается стрелкой слева от заголовка. Если стрелка скрыта, используйте меню View → Collapse Selected Heading. В старом Notebook 6 сворачивание заголовков требует расширения Collapsible Headings.


## Постановка

В квадрате $0\le x,z\le1$ ищем стационарную температуру:
$$
\Delta u=f,\qquad f=-80.
$$
Граничные условия:
$$
u(0,z)=u(1,z)=20+20z(1-z),\qquad
u(x,0)=u(x,1)=20+20x(1-x).
$$
Точное решение для проверки:
$$
u_{\text{exact}}(x,z)=20+20[x(1-x)+z(1-z)].
$$
Обе вторые производные равны $-40$, поэтому их сумма равна $-80$. Это равномерный внутренний нагрев и отвод тепла через границу. Центр имеет температуру 30, углы — 20, середины сторон — 25.

Численные алгоритмы используют только $f$ и граничные условия. Точное решение применяется отдельно для вычисления ошибок.


In [ ]:
import math
import random
import sys
from pathlib import Path
from io import StringIO
from contextlib import redirect_stdout

# Работает при запуске Jupyter из корня проекта или из папки src.
module_folder = Path.cwd()
if not (module_folder / "numerical_common.py").exists():
    module_folder = module_folder / "src"
if str(module_folder) not in sys.path:
    sys.path.insert(0, str(module_folder))

from numerical_common import (
    residual, calculate_errors
)
from settling import solve_settling
from monte_carlo import estimate_point, solve_monte_carlo
from notebook_output import show_output

from task1_problem import exact_temperature, source, boundary_temperature
from monte_carlo_parallel import solve_monte_carlo_parallel
import os

intervals = 10
h = 1.0 / intervals
tau = 0.02
tolerance = 0.000001
max_iterations = 10000
particle_counts = [100, 1000, 10000, 100000]

x_values = []
z_values = []
exact_field = []
for n in range(intervals + 1):
    x_values.append(n * h)
    z_values.append(n * h)
    row = []
    for k in range(intervals + 1):
        row.append(exact_temperature(k * h, n * h))
    exact_field.append(row)

output_text = StringIO()
with redirect_stdout(output_text):
    print("Сетка:", intervals + 1, "на", intervals + 1)
    print("Температура в центре:", exact_field[intervals // 2][intervals // 2])
show_output("Параметры сетки", output_text.getvalue())


## Метод установления: продольно-поперечная схема

Вводим искусственное время:
$$
u_t=L_xu+L_zu-f.
$$
Здесь $L_x,L_z$ — центральные разностные вторые производные. При установлении $u_t=0$ получается исходное уравнение. В описании задания в одной разностной формуле стоит $+f$; для принятой записи $\Delta u=f$ нужен знак $-f$.

Используем схему Писмена — Рэкфорда, один из вариантов продольно-поперечного метода:
$$
\frac{y^{(1/2)}-y}{\tau/2}=L_xy^{(1/2)}+L_zy-f,
$$
$$
\frac{\hat y-y^{(1/2)}}{\tau/2}=L_xy^{(1/2)}+L_z\hat y-f.
$$
Источник входит в оба полушага: каждый длится $\tau/2$. Сначала решаем отдельные строки по $x$, затем отдельные столбцы по $z$.

Как в `DESCRIPTION.md`, $y$ — старый слой, $\hat y$ — новый слой. Промежуточное значение обозначаем $y^{(1/2)}$: звёздочку для него не используем. В коде это `field`, `new_field` и `middle`. Операторы $L_x,L_z$ в этих формулах уже являются разностными.

Обозначим $r=\tau/(2h^2)$. Для первой системы:
$$
-r y^{(1/2)}_{n,k-1}+(1+2r)y^{(1/2)}_{n,k}-r y^{(1/2)}_{n,k+1}
=y_{n,k}+r(y_{n-1,k}-2y_{n,k}+y_{n+1,k})-\frac\tau2 f_{n,k}.
$$
Для второй:
$$
-r \hat y_{n-1,k}+(1+2r)\hat y_{n,k}-r \hat y_{n+1,k}
=y^{(1/2)}_{n,k}+r(y^{(1/2)}_{n,k-1}-2y^{(1/2)}_{n,k}+y^{(1/2)}_{n,k+1})-\frac\tau2 f_{n,k}.
$$
Известные граничные значения переносим вправо. Обе системы трёхдиагональные.


### Прогонка

Для системы $a_i v_{i-1}+b_i v_i+c_i v_{i+1}=d_i$ вводим $v_i=P_i v_{i+1}+Q_i$:
$$
P_i=\frac{-c_i}{b_i+a_iP_{i-1}},\qquad
Q_i=\frac{d_i-a_iQ_{i-1}}{b_i+a_iP_{i-1}}.
$$
В первой строке $P_0=-c_0/b_0$, $Q_0=d_0/b_0$. После прямого прохода выполняем обратную подстановку.

Останавливаемся по невязке исходного уравнения:
$$
R=\max_{n,k\text{ внутри}}|L_xy+L_zy-f|<\varepsilon.
$$


In [2]:
output_text = StringIO()
with redirect_stdout(output_text):
    settling_field, iterations, settling_time, history = solve_settling(
        intervals, tau, tolerance, max_iterations, source, boundary_temperature, 20
    )
    print("Итераций:", iterations)
    print("Время, с:", settling_time)
    print("Конечная невязка:", history[len(history) - 1])
show_output("Результат метода установления", output_text.getvalue())


## Метод Монте-Карло

Из разностного уравнения:
$$
u_{n,k}=\frac{u_{n,k-1}+u_{n,k+1}+u_{n-1,k}+u_{n+1,k}}4-\frac{h^2}{4}f_{n,k}.
$$
Коэффициенты $1/4$ задают вероятности четырёх направлений. Запускаем путь из узла, где ищем температуру. В каждом внутреннем узле прибавляем $-h^2f/4$, выбираем случайного соседа и продолжаем путь до первого попадания на границу.

При $f=-80$ каждая посещённая внутренняя точка добавляет $20h^2$. Результат пути длины $s$:
$$
T_\alpha=g(\text{точка выхода})+20h^2s.
$$
Искомое значение оценивается средним **полных результатов** $N$ независимых путей:
$$
\widehat u=\frac1N\sum_{\alpha=1}^NT_\alpha.
$$
Длину пути искусственно не ограничиваем: усечение изменило бы ответ. В конечной сетке выход происходит с вероятностью 1.

Для оценки неопределённости считаем выборочную дисперсию $S^2$ результатов путей и стандартную ошибку среднего:
$$
S^2=\frac{\sum_\alpha(T_\alpha-\overline T)^2}{N-1},\qquad
SE=\sqrt{S^2/N}.
$$
Ожидаемый масштаб случайной ошибки — $1/\sqrt N$. У отдельного запуска ошибка может увеличиться при росте $N$, поэтому ниже также повторим опыт в центре с разными seed.

Рассчитываем всё поле, а не только центр. Время метода включает генерацию случайных чисел и все шаги траекторий.


## Сравнение точности и времени

Ошибки считаем только по внутренним узлам, поскольку на границе значения заданы точно:
$$
E_\infty=\max|y-u_{\text{exact}}|,\qquad
RMSE=\sqrt{\frac1M\sum_{n,k\text{ внутри}}(y_{n,k}-u_{\text{exact}}(x_k,z_n))^2}.
$$
$M$ — число внутренних узлов. Сравниваем полные поля на одинаковой сетке. Время построения графиков не входит во время решения. Это одиночные замеры на текущем компьютере, а не универсальная оценка производительности.

У выбранной квадратичной функции центральные разности дают точные вторые производные. Поэтому ошибки сеточной аппроксимации здесь нет (кроме округления); тест проверяет реализацию, завершённость установления и статистическую ошибку. Для изучения порядка точности по $h$ нужен дополнительный неквадратичный тест.


In [3]:
output_text = StringIO()
with redirect_stdout(output_text):
    settling_max_error, settling_rmse = calculate_errors(settling_field, intervals, exact_temperature)
    mc_fields = []
    mc_standard_errors = []
    mc_times = []
    mc_steps = []
    mc_max_errors = []
    mc_rmse_values = []
    
    print("Метод             N     Время, с       E_max         RMSE")
    print("Установление      -    %10.4f   %12.6g   %12.6g" %
          (settling_time, settling_max_error, settling_rmse))
    
    for i in range(len(particle_counts)):
        particles = particle_counts[i]
        field, standard_errors, elapsed, steps = solve_monte_carlo(
            intervals, particles, 12345, source, boundary_temperature
        )
        maximum, rmse = calculate_errors(field, intervals, exact_temperature)
        mc_fields.append(field)
        mc_standard_errors.append(standard_errors)
        mc_times.append(elapsed)
        mc_steps.append(steps)
        mc_max_errors.append(maximum)
        mc_rmse_values.append(rmse)
        print("Монте-Карло  %6d    %10.4f   %12.6g   %12.6g" %
              (particles, elapsed, maximum, rmse))
        print("  Число шагов всех траекторий:", steps)
    
    center = intervals // 2
    last = len(particle_counts) - 1
    center_value = mc_fields[last][center][center]
    center_se = mc_standard_errors[last][center][center]
    print("\nЦентр: точное значение =", exact_temperature(center * h, center * h))
    print("Установление =", settling_field[center][center])
    print("Монте-Карло =", center_value, "; SE =", center_se)
    print("Приближённый 95% интервал в центре:",
          center_value - 1.96 * center_se, center_value + 1.96 * center_se)
    print("Время МК / время установления =", mc_times[last] / settling_time)
show_output("Сравнение методов и время расчёта", output_text.getvalue())


KeyboardInterrupt: 

### Повторные опыты в центре

Для каждого $N$ выполняем 10 независимых оценок центральной температуры. Считаем RMSE этих оценок относительно 30. Время этого отдельного эксперимента не добавляется в таблицу полного поля. Приближённый 95% интервал выше относится только к одной точке и опирается на нормальное приближение; это не гарантия точности всего поля.


In [ ]:
output_text = StringIO()
with redirect_stdout(output_text):
    repeats = 10
    repeat_rmse = []
    for i in range(len(particle_counts)):
        particles = particle_counts[i]
        squared_sum = 0.0
        for repeat in range(repeats):
            generator = random.Random(5000 + repeat)
            value, error, steps = estimate_point(
                center, center, intervals, particles, generator, source, boundary_temperature
            )
            difference = value - exact_temperature(center * h, center * h)
            squared_sum = squared_sum + difference * difference
        rmse = math.sqrt(squared_sum / repeats)
        repeat_rmse.append(rmse)
        print("N =", particles, "; RMSE по", repeats, "повторам =", rmse)
    
    # Проверка граничных условий и результата установления.
    assert residual(settling_field, intervals, source) < tolerance
    assert settling_max_error < 0.000001
    for i in range(len(mc_fields)):
        for n in range(intervals + 1):
            for k in range(intervals + 1):
                if n == 0 or n == intervals or k == 0 or k == intervals:
                    expected = boundary_temperature(k * h, n * h)
                    assert abs(mc_fields[i][n][k] - expected) < 0.000000001
    print("Проверки невязки, точности установления и границ пройдены.")
show_output("Повторные опыты и проверки", output_text.getvalue())


## Параллельный метод Монте-Карло

Независимые внутренние узлы сетки распределяем между 20 процессами. На сетке 11×11 есть 81 внутренний узел, поэтому заданий достаточно для всех процессов. Каждое задание содержит все траектории одного узла; после завершения узла процесс получает следующий. Внутри каждого процесса используются те же функции `walk` и `estimate_point`. Процессы позволяют одновременно выполнять Python-циклы на разных ядрах.

Функции исходных данных вынесены в `task1_problem.py`, а работа с процессами — в `monte_carlo_parallel.py`. Это необходимо для запуска через `spawn` из Jupyter: дочерние процессы импортируют функции из Python-модулей.

Для каждого узла задаём отдельный seed:
$$
seed_{n,k}=seed+n(K+1)+k,
$$
где $K$ — число интервалов. Последовательный и параллельный методы используют одинаковые траектории в каждом узле. Поэтому поля, стандартные ошибки и общее число шагов должны совпасть точно, независимо от порядка выполнения процессов. Seed по узлам теперь используется и в последовательной версии; старые сохранённые результаты могут отличаться.

Сравниваем время расчёта **всего поля** при одинаковых $N$ и сетке. Параллельное время включает запуск процессов, передачу заданий, сбор результатов и завершение пула. Для каждого $N$ пул создаётся заново. На маленьком объёме накладные расходы могут сделать параллельную версию медленнее.

Ускорение и эффективность:
$$
S_p=\frac{t_{seq}}{t_{par}},\qquad E_p=\frac{S_p}{p}.
$$
$p$ — число рабочих процессов. Значение $S_p>1$ означает ускорение. Результаты зависят от доступных ядер и нагрузки компьютера; один замер не заменяет статистику повторных замеров.

Список `particle_counts` сохранён: он включает до миллиона траекторий на каждый узел. Полный запуск может занимать много времени. Для быстрой проверки можно вручную оставить `[100, 1000, 10000]`.


In [ ]:
workers = 20

parallel_fields = []
parallel_times = []
parallel_rmse = []
speedups = []
efficiencies = []

output_text = StringIO()
with redirect_stdout(output_text):
    print("Число рабочих процессов:", workers)
    print("N         Послед., с   Паралл., с   Ускорение   Эффективность")
    for i in range(len(particle_counts)):
        particles = particle_counts[i]
        field, errors, elapsed, steps = solve_monte_carlo_parallel(
            intervals, particles, 12345, source, boundary_temperature, workers
        )
        # Одинаковые seed гарантируют одинаковые результаты в каждом узле.
        assert field == mc_fields[i]
        assert errors == mc_standard_errors[i]
        assert steps == mc_steps[i]
        maximum, rmse = calculate_errors(field, intervals, exact_temperature)
        speedup = mc_times[i] / elapsed
        efficiency = speedup / workers
        parallel_fields.append(field)
        parallel_times.append(elapsed)
        parallel_rmse.append(rmse)
        speedups.append(speedup)
        efficiencies.append(efficiency)
        print("%8d  %11.4f  %11.4f  %9.3f  %12.3f" %
              (particles, mc_times[i], elapsed, speedup, efficiency))
    print("Поля, стандартные ошибки и число шагов совпадают для всех N.")
show_output("Последовательный и параллельный Монте-Карло", output_text.getvalue())


### Время, ускорение и точность параллельной версии

Три графика объединены в одну строку. При одинаковых траекториях ошибки обеих версий совпадают, различается время вычисления.

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 3, figsize=(18, 5), layout="constrained")
axes[0].loglog(particle_counts, mc_times, "o-", label="Последовательный")
axes[0].loglog(particle_counts, parallel_times, "s-", label="Параллельный")
axes[0].set_title("Время полного расчёта")
axes[0].set_xlabel("N на узел")
axes[0].set_ylabel("Время, с")
axes[0].legend()

axes[1].semilogx(particle_counts, speedups, "o-", label="Измеренное ускорение")
axes[1].axhline(1, color="gray", linestyle="--", label="Без ускорения")
axes[1].axhline(workers, color="green", linestyle=":", label="Ориентир p")
axes[1].set_title("Ускорение на " + str(workers) + " процессах")
axes[1].set_xlabel("N на узел")
axes[1].set_ylabel("Последовательное время / параллельное")
axes[1].legend()

axes[2].loglog(mc_times, mc_rmse_values, "o-", label="Последовательный")
axes[2].loglog(parallel_times, parallel_rmse, "s-", label="Параллельный")
axes[2].set_title("Точность и время")
axes[2].set_xlabel("Время, с")
axes[2].set_ylabel("RMSE")
axes[2].legend()
for i in range(3):
    axes[i].grid(True)
plt.show()


### Результат проверки параллельной версии

Предыдущий контрольный запуск на сетке 11×11 с четырьмя процессами и распределением по строкам (исторические результаты, не замеры новой версии на 20 процессах):

| N на узел | Последовательно, с | Параллельно, с | Ускорение | Эффективность |
|---:|---:|---:|---:|---:|
| 100 | 0,0458 | 0,0635 | 0,720 | 0,180 |
| 1000 | 0,4855 | 0,2070 | 2,345 | 0,586 |
| 10000 | 5,0851 | 1,5643 | 3,251 | 0,813 |

Проверено точное совпадение полей, стандартных ошибок и числа шагов. При маленьком N накладные расходы превышают выигрыш от распараллеливания; при N=10000 получено ускорение около 3,25 раза. Это результаты конкретного контрольного запуска, а не гарантированное ускорение. Значения 100000 и 1000000 в этой проверке не запускались. При выполнении ноутбука таблица вывода и графики выше обновятся; эта статическая таблица описывает контрольный запуск.


## Графики Matplotlib

Сравниваемые графики объединены в общие фигуры с одной строкой подграфиков: три поверхности, два поля ошибки и четыре диагностических графика. Разные масштабы ошибок показаны отдельными цветовыми шкалами.

### Решения на одной сетке

In [ ]:
figure, axes = plt.subplots(1, 4, figsize=(23, 5), layout="constrained")

axes[0].plot(x_values, exact_field[center], label="Точное")
axes[0].plot(x_values, settling_field[center], "o", label="Установление")
axes[0].plot(x_values, mc_fields[last][center], "s--", label="Монте-Карло")
axes[0].set_title("Сечение z = 0,5")
axes[0].set_xlabel("x")
axes[0].set_ylabel("Температура")
axes[0].legend()

iteration_numbers = []
for i in range(len(history)):
    iteration_numbers.append(i + 1)
axes[1].semilogy(iteration_numbers, history)
axes[1].set_title("Установление: невязка")
axes[1].set_xlabel("Итерация")
axes[1].set_ylabel("Максимальная невязка")

reference_errors = []
for i in range(len(particle_counts)):
    reference_errors.append(
        repeat_rmse[0] * math.sqrt(particle_counts[0] / particle_counts[i])
    )
axes[2].loglog(particle_counts, mc_rmse_values, "o-", label="Полное поле")
axes[2].loglog(particle_counts, repeat_rmse, "s-", label="Центр: 10 повторов")
axes[2].loglog(particle_counts, reference_errors, "--", label="1 / sqrt(N)")
axes[2].set_title("Ошибка Монте-Карло")
axes[2].set_xlabel("N на узел")
axes[2].set_ylabel("RMSE")
axes[2].legend()

axes[3].loglog(mc_times, mc_rmse_values, "o-", label="Монте-Карло")
axes[3].loglog([settling_time], [settling_rmse], "s", label="Установление")
for i in range(len(particle_counts)):
    axes[3].annotate("N=" + str(particle_counts[i]),
                     (mc_times[i], mc_rmse_values[i]))
axes[3].set_title("Точность и время")
axes[3].set_xlabel("Время, с")
axes[3].set_ylabel("RMSE")
axes[3].legend()
for i in range(4):
    axes[i].grid(True)
plt.show()


## Выводы

<details>
<summary><strong>Выводы — нажмите, чтобы раскрыть</strong></summary>
<ol>
<li>Оба алгоритма решают одну дискретную задачу с одинаковыми границами и источником. Метод установления получает ответ последовательными временными шагами; Монте-Карло — усреднением результатов случайных путей.</li>
<li>Установление завершается по невязке. Полученную ошибку и число итераций смотрим в таблице и на графике. Изменение τ влияет на скорость сходимости.</li>
<li>Для Монте-Карло рост N увеличивает время примерно пропорционально числу путей. Типичный статистический масштаб ошибки уменьшается как 1/√N; отдельные результаты могут колебаться.</li>
<li>Для всего поля на небольшой двумерной сетке метод установления ожидаемо выгоднее. Конкретное отношение времён вычислено выше. Нельзя приписывать Монте-Карло выигрыш от расчёта одной точки, сравнивая его с временем расчёта всего поля.</li>
<li>Самое простое ускорение Монте-Карло — независимый параллельный расчёт траекторий. Также полезны пакетная обработка и расчёт только нужных точек. В этой учебной реализации сохранены обычные циклы; сравнение показывает базовые алгоритмы без таких оптимизаций.</li>
<li>Для полного исследования можно изменить <code>intervals</code>, <code>tau</code>, <code>particle_counts</code> и повторно запустить все ячейки. Измельчение сетки увеличивает число узлов и среднюю длину блуждания, поэтому Монте-Карло быстро дорожает.</li>
</ol>
</details>

### Полученные результаты контрольного запуска

Вычислительные ячейки проверены на Python 3; их выводы сохранены выше. Графические ячейки используют Matplotlib. Таблица ниже относится к исходному контрольному запуску вычислительных методов.

| Метод | N на узел | Время, с | Максимальная ошибка | RMSE |
|---|---:|---:|---:|---:|
| Установление | — | 0,0048 | 3,85·10⁻⁸ | 2,14·10⁻⁸ |
| Монте-Карло | 100 | 0,0472 | 0,884 | 0,3431 |
| Монте-Карло | 1000 | 0,5092 | 0,2764 | 0,1078 |
| Монте-Карло | 10000 | 4,9011 | 0,10692 | 0,03480 |

Установление достигло невязки 7,54·10⁻⁷ за 50 итераций. При увеличении числа траекторий в 100 раз RMSE Монте-Карло уменьшилась примерно в 9,9 раза, что согласуется с масштабом 1/√N. При N=10000 расчёт всего поля методом Монте-Карло оказался примерно в 1029 раз медленнее установления при существенно большей ошибке. Это сравнение конкретных настроек: методы здесь не доводились до одинаковой точности. Времена зависят от компьютера; повторный запуск обновит выводы ячеек, но эта записанная таблица остаётся описанием контрольного запуска.
